<a href="https://colab.research.google.com/github/dohuongtra2310-arch/Stock-Trading-Backtester/blob/main/backtest_logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install yfinance plotly -q  # Cài đặt thư viện âm thầm
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go

In [23]:
# --- 1. Data Acquisition & Cleaning ---
ticker = 'IVV.AX'
# Download historical data for the last 2 years
df = yf.download(ticker, period='2y')

# Handle MultiIndex columns (common in recent yfinance versions)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Select only the 'Close' price and create a copy to avoid SettingWithCopyWarning
data = df[['Close']].copy()

# --- 2. Technical Indicator Calculation ---
# Calculate 20-day Simple Moving Average (Fast Signal)
data['SMA20'] = data['Close'].rolling(window=20).mean()
# Calculate 50-day Simple Moving Average (Slow Signal)
data['SMA50'] = data['Close'].rolling(window=50).mean()

# Drop rows with NaN values generated during SMA calculation
data = data.dropna()

# --- 3. Backtesting Logic ---
# Initial capital setup
cash = 10000
shares = 0
data['Total_Value'] = 0.0

for i in range(len(data)):
    # Buy Signal: Golden Cross (SMA20 crosses above SMA50) while out of market
    if data['SMA20'].iloc[i] > data['SMA50'].iloc[i] and shares == 0:
        shares = cash / data['Close'].iloc[i]
        cash = 0
        print(f"Buy at {data.index[i].date()} - Price: {data['Close'].iloc[i]:.2f}")

    # Sell Signal: Death Cross (SMA20 crosses below SMA50) while holding shares
    elif data['SMA20'].iloc[i] < data['SMA50'].iloc[i] and shares > 0:
        cash = shares * data['Close'].iloc[i]
        shares = 0
        print(f"Sell at {data.index[i].date()} - Price: {data['Close'].iloc[i]:.2f}")

    # Update daily Net Asset Value (NAV)
    data.iloc[i, data.columns.get_loc('Total_Value')] = cash + (shares * data['Close'].iloc[i])

# Calculate final performance metrics
final_value = data['Total_Value'].iloc[-1]
profit = ((final_value - 10000) / 10000) * 100

print(f"\n--- Strategy Result ---")
print(f"Final Portfolio Value: ${final_value:.2f}")
print(f"Total Return: {profit:.2f}%")

# --- 4. Performance Benchmarking (Buy & Hold Comparison) ---
buy_price = data['Close'].iloc[0]
sell_price = data['Close'].iloc[-1]
buy_hold_return = ((sell_price - buy_price) / buy_price) * 100

print(f"\n--- Final Comparison ---")
print(f"Strategy Return (SMA Crossover): {profit:.2f}%")
print(f"Buy & Hold Return: {buy_hold_return:.2f}%")

# Calculate Alpha (Excess return generated by the strategy)
alpha = profit - buy_hold_return
if alpha > 0:
    print(f"Conclusion: Strategy BEAT the market by {alpha:.2f}%! 🚀")
else:
    print(f"Conclusion: Market outperformed the strategy by {abs(alpha):.2f}%. 📉")

# --- 5. Interactive Visualization ---
fig = go.Figure()
# Plot Price and Moving Averages
fig.add_trace(go.Scatter(x=data.index, y=data['Close'], name='Price', line=dict(color='black', width=1)))
fig.add_trace(go.Scatter(x=data.index, y=data['SMA20'], name='SMA 20 (Fast)', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=data.index, y=data['SMA50'], name='SMA 50 (Slow)', line=dict(color='red')))

# Update layout for a professional look
fig.update_layout(
    title=f'Backtesting Results for {ticker}',
    xaxis_title='Date',
    yaxis_title='Price (AUD)',
    template='plotly_white',
    hovermode='x unified'
)
fig.show()

/tmp/ipykernel_22292/757619089.py:4: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed


Buy at 2024-07-10 - Price: 54.06
Sell at 2024-08-20 - Price: 54.60
Buy at 2024-09-06 - Price: 53.46
Sell at 2025-03-05 - Price: 61.23
Buy at 2025-05-20 - Price: 60.97
Sell at 2025-12-16 - Price: 67.83
Buy at 2025-12-17 - Price: 68.08
Sell at 2025-12-18 - Price: 67.70
Buy at 2026-04-22 - Price: 65.96

--- Strategy Result ---
Final Portfolio Value: $12979.96
Total Return: 29.80%

--- Final Comparison ---
Strategy Return (SMA Crossover): 29.80%
Buy & Hold Return: 23.77%
Conclusion: Strategy BEAT the market by 6.03%! 🚀
